## Import Packages

In [1]:
import pandas as pd
import ast
import re
from datasets import load_dataset

## Extract arXiv IDs

Since we are joining the datasets between OpenAlex and Hugging Face, each research paper's respective id do not match, except for their arXiv id. Hence, we will extract the arXiv ids from **OpenAlex**'s papers by finding their associated URL in the `locations` column.

In [17]:
# Load OpenAlex dataset
openalex_df = pd.read_parquet("https://www.dropbox.com/scl/fi/6uyg2v5we0nvh591y7dj8/openalex_works.parquet?rlkey=7pi1gq2pabf48t4hihncfiazj&st=tae6bjyy&dl=1")

def extract_arxiv_id(locations):
    if pd.isna(locations):
        return None

    # Convert CSV string back into a Python list
    if isinstance(locations, str):
        locations = ast.literal_eval(locations)

    for location in locations:
        url = location.get("landing_page_url") or location.get("pdf_url") or ""

        arxiv_url = re.search(r"arxiv\.org/(?:abs|pdf)/([^?#]+)", url)
        if arxiv_url is not None:
            arxiv_id = arxiv_url.group(1)
            return re.sub(r"v\d+$", "", arxiv_id)
    return None

openalex_df["arxiv_id"] = openalex_df["locations"].apply(extract_arxiv_id)

# Remove papers with missing and/or duplicated arXiv IDs
openalex_df.dropna(subset=["arxiv_id"], inplace=True)
openalex_df.drop_duplicates(subset=["arxiv_id"], inplace=True)

openalex_ids = openalex_df["arxiv_id"]
print(f"Extracted {len(openalex_ids):,} arXiv IDs")

Extracted 122,136 arXiv IDs


## Match Papers from Hugging Face

In [ ]:
# Load the (entire) arXiv dataset from Hugging Face
dataset = load_dataset("open-index/open-arxiv", split="train", streaming=True)

arxiv_records = []

for paper in dataset:
    arxiv_id = re.sub(r"v\d+$", "", str(paper["id"]))

    if arxiv_id in openalex_ids:
        arxiv_records.append({"arxiv_id": arxiv_id,
                              "categories": paper["categories"],
                              "comments": paper["comments"]})

arxiv_df = pd.DataFrame(arxiv_records)

# Remove papers with duplicate arXiv IDs
arxiv_df.drop_duplicates(subset=["arxiv_id"], inplace=True)

print(f"Matched {len(arxiv_df):,} papers")

README.md:   0%|          | 0.00/19.9k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/417 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/417 [00:00<?, ?it/s]

KeyboardInterrupt: 

**Note**: Haven't run the above cell to find the arXiv `categories` and `comments` since the dataset contains around 3 million records, which will take too long to run.

## Extract Papers with Code

In [3]:
# Load Papers with Code dataset
dataset_code = load_dataset("pwc-archive/links-between-paper-and-code", split="train")

pwc_df = dataset_code.to_pandas()

# Normalize arXiv IDs to be consistent
def normalize_arxiv_id(arxiv_id):
    if pd.isna(arxiv_id):
        return None

    arxiv_id = str(arxiv_id).strip()
    arxiv_id = re.sub(r"\.pdf$", "", arxiv_id)
    arxiv_id = re.sub(r"v\d+$", "", arxiv_id)

    return arxiv_id

pwc_df["arxiv_id"] = pwc_df["paper_arxiv_id"].apply(normalize_arxiv_id)

# Remove papers with no repo_url
pwc_df = pwc_df[pwc_df["repo_url"].str.strip() != ""]

# Remove nulls and duplicate using (arxiv_id, repo_url) pairs
pwc_df.dropna(subset=["arxiv_id", "repo_url"], inplace=True)
pwc_df.drop_duplicates(subset=["arxiv_id", "repo_url"], inplace=True)

# Group by each paper's unique arXiv id
pwc_features = pwc_df.groupby("arxiv_id").agg(num_repositories=("repo_url", "nunique")).reset_index()

# Create binary feature (Every paper appears here has code)
pwc_features["has_code"] = 1

print(f"Papers with code: {len(pwc_features):,}")

README.md:   0%|          | 0.00/826 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 41.4MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/300161 [00:00<?, ? examples/s]

Papers with code: 207,415


**Note**: A paper without a Papers with Code match does not necessarily mean it has no publicly available code. It only means the archive does not contain a matching repository record. Therefore, `has_code` indicates whether a repository is linked in the Papers with Code archive, rather than whether code actually exists.

Also, repository links may have been added after the paper's initial publication.

## Join Dataset

In [4]:
# Left join to preserve all OpenAlex papers
# final_df = openalex_df.merge(arxiv_df, on="arxiv_id",how="left")

# Join Papers with Code features
# final_df = final_df.merge(pwc_features, on="arxiv_id", how="left")
final_df = openalex_df.merge(pwc_features, on="arxiv_id", how="left")

# Papers without repository matches receive zero
final_df["has_code"] = final_df["has_code"].fillna(0).astype(int)
final_df["num_repositories"] = (final_df["num_repositories"].fillna(0).astype(int))

# Remove any nulls
final_df.dropna(inplace=True)

print(f"Final dataset: {len(final_df):,} papers")
print(f"Papers with code: {final_df['has_code'].sum():,}")

print(final_df.info())

Final dataset: 108,228 papers
Papers with code: 25,791
<class 'pandas.core.frame.DataFrame'>
Index: 108228 entries, 0 to 122134
Data columns (total 23 columns):
 #   Column                       Non-Null Count   Dtype 
---  ------                       --------------   ----- 
 0   id                           108228 non-null  object
 1   doi                          108228 non-null  object
 2   title                        108228 non-null  object
 3   abstract_inverted_index      108228 non-null  object
 4   publication_date             108228 non-null  object
 5   publication_year             108228 non-null  int64 
 6   counts_by_year               108228 non-null  object
 7   authorships                  108228 non-null  object
 8   institutions_distinct_count  108228 non-null  int64 
 9   countries_distinct_count     108228 non-null  int64 
 10  referenced_works_count       108228 non-null  int64 
 11  referenced_works             108228 non-null  object
 12  primary_topic         

## Data Features

The dataset contains metadata for **computer science** research papers indexed in **OpenAlex** and associated with **arXiv** publications from 2018–2020. It was then joined with the dataset from **Hugging Face** containing arXiv papers with at least one linked code repositories.

- `id` = Unique **OpenAlex** identifier assigned to each research paper
- `doi` = Digital Object Identifier, a unique identifier used to locate and reference a published paper (some papers may not have a doi)
- `title` = Title of the research paper
- `abstract_inverted_index` = **OpenAlex**'s way of storing a paper's *abstract* in a word-to-position mapping, rather than storing the abstract as one continuous text string (e.g., "I like sushi." --> {'I': [0], 'like': [1], 'sushi.': [2]})
- `publication_date` = Full publication date of the paper, including year, month, and day
- `publication_year` = Calendar year in which the paper was published
- `counts_by_year` = Number of citation received by the paper in each calendar year, as recorded by **OpenAlex** --> ***Target***
- `authorships` = Detailed authorship information, including the paper's authors and their associated institutions and countries
- `institutions_distinct_count` = Number of unique institutions represented among the paper's authors
- `countries_distinct_count` = Number of unique countries represented among the paper's authors
- `referenced_works_count` = Total number of other research papers cited by the current paper
- `referenced_works` = A list of **OpenAlex** identifiers corresponding to the papers cited in the current paper
- `primary_topic` = Main research topic assigned to the paper by **OpenAlex**, including its associated field, subfield, and domain
- `topics` = List of research topics associated with the paper, providing a broader representation of its research areas
- `open_access` = Information about the paper's open-access status, including whether it's open accessible and its acess designation
- `locations` = A list of locations where the paper is *available*, including publication sources and repository records
- `locations_count` = Number of distinct locations associated with the paper in **OpenAlex**
- `primary_location` = Main publication or repository location associated with the paper, including information about its source (e.g., OpenAlex)
- `type` = **OpenAlex** classification of the work, such as 'article', 'preprint', or 'book chapter'
- `language` = Language in which the paper is written, represented using a language code (e.g., 'en' for "english")
- `arxiv_id` = Unique **arXiv** identifier associated with the paper, used to match records with external arXiv-related datasets
- `num_repositories` = Number of distinct code repositories associated with the paper in the **Papers with Code** dataset
- `has_code` = Binary indicator of whether the paper has at least one code repository in the **Papers with Code** dataset

**Note**: Code availability and publication-location information may have been updated *after* the paper's original publication. Their use as predictors should therefore be evaluated carefully to avoid temporal data leakage.

**Idea**: `abstract_inverted_index` can be used to engineer features like `abstract_length`.

**Idea**: Remove `primary_location` since we match all the arXiv research papers from **Papers with Code** with those from **OpenAlex**, so this column will all be **OpenAlex**.

In [ ]:
final_df.head(5)

,id,doi,title,abstract_inverted_index,publication_date,publication_year,counts_by_year,authorships,institutions_distinct_count,countries_distinct_count,...,topics,open_access,locations,locations_count,primary_location,type,language,arxiv_id,num_repositories,has_code
0,https://openalex.org/W3003257820,https://doi.org/10.1038/s41592-019-0686-2,SciPy 1.0: fundamental algorithms for scientif...,"{'SciPy': [0, 18, 65], 'is': [1], 'an': [2, 56...",2020-02-03,2020,"[{'year': 2026, 'cited_by_count': 7041}, {'yea...","[{'author_position': 'first', 'author': {'id':...",84,18,...,"[{'id': 'https://openalex.org/T13650', 'displa...","{'is_oa': True, 'oa_status': 'hybrid', 'oa_url...","[{'id': 'doi:10.1038/s41592-019-0686-2', 'doi'...",18,"{'id': 'doi:10.1038/s41592-019-0686-2', 'doi':...",review,en,1907.10121,1,1
1,https://openalex.org/W2963163009,https://doi.org/10.1109/cvpr.2018.00474,MobileNetV2: Inverted Residuals and Linear Bot...,"{'In': [0], 'this': [1, 133, 143], 'paper': [2...",2018-06-01,2018,"[{'year': 2026, 'cited_by_count': 3194}, {'yea...","[{'author_position': 'first', 'author': {'id':...",1,1,...,"[{'id': 'https://openalex.org/T10036', 'displa...","{'is_oa': True, 'oa_status': 'green', 'oa_url'...","[{'id': 'doi:10.1109/cvpr.2018.00474', 'doi': ...",3,"{'id': 'doi:10.1109/cvpr.2018.00474', 'doi': '...",conference-paper,en,1801.04381,159,1
2,https://openalex.org/W3035965352,https://doi.org/10.1038/s41586-020-2649-2,Array programming with NumPy,"{'Abstract': [0], 'Array': [1], 'programming':...",2020-09-16,2020,"[{'year': 2026, 'cited_by_count': 4947}, {'yea...","[{'author_position': 'first', 'author': {'id':...",18,7,...,"[{'id': 'https://openalex.org/T13650', 'displa...","{'is_oa': True, 'oa_status': 'hybrid', 'oa_url...","[{'id': 'doi:10.1038/s41586-020-2649-2', 'doi'...",8,"{'id': 'doi:10.1038/s41586-020-2649-2', 'doi':...",review,en,2006.10256,1,1
5,https://openalex.org/W2962785568,https://doi.org/10.1109/cvpr.2018.00068,The Unreasonable Effectiveness of Deep Feature...,"{'While': [0], 'it': [1], 'is': [2, 145, 173],...",2018-06-01,2018,"[{'year': 2026, 'cited_by_count': 2743}, {'yea...","[{'author_position': 'first', 'author': {'id':...",2,1,...,"[{'id': 'https://openalex.org/T11165', 'displa...","{'is_oa': True, 'oa_status': 'green', 'oa_url'...","[{'id': 'doi:10.1109/cvpr.2018.00068', 'doi': ...",3,"{'id': 'doi:10.1109/cvpr.2018.00068', 'doi': '...",conference-paper,en,1801.03924,24,1
6,https://openalex.org/W3035524453,https://doi.org/10.1109/cvpr42600.2020.00975,Momentum Contrast for Unsupervised Visual Repr...,"{'We': [0], 'present': [1], 'Momentum': [2], '...",2020-06-01,2020,"[{'year': 2026, 'cited_by_count': 1298}, {'yea...","[{'author_position': 'first', 'author': {'id':...",1,1,...,"[{'id': 'https://openalex.org/T11307', 'displa...","{'is_oa': True, 'oa_status': 'green', 'oa_url'...","[{'id': 'doi:10.1109/cvpr42600.2020.00975', 'd...",3,"{'id': 'doi:10.1109/cvpr42600.2020.00975', 'do...",conference-paper,en,1911.05722,44,1


## Test

In [16]:
locations = ast.literal_eval(final_df['locations'][0])
print(type(locations))

for location in locations:
    print(location)
#         url = location.get("landing_page_url") or location.get("pdf_url") or ""

#         match = re.search(r"arxiv\.org/(?:abs|pdf)/([^?#]+)", url)
#         if match:
#             arxiv_id = match.group(1)
#             print(re.sub(r"v\d+$", "", arxiv_id))

<class 'list'>
{'id': 'doi:10.1038/s41592-019-0686-2', 'doi': 'https://doi.org/10.1038/s41592-019-0686-2', 'is_oa': True, 'landing_page_url': 'https://doi.org/10.1038/s41592-019-0686-2', 'pdf_url': None, 'source': {'id': 'https://openalex.org/S127827428', 'display_name': 'Nature Methods', 'issn_l': '1548-7091', 'issn': ['1548-7091', '1548-7105'], 'is_oa': False, 'is_in_doaj': False, 'is_core': True, 'listed_in': ['cwts-core', 'dongbi-a', 'doyens', 'jufo-3', 'ki-jl-3', 'medline', 'norway-2', 'poland-200', 'russia-white-list-1', 'vabb-shw'], 'host_organization': 'https://openalex.org/P4310319908', 'host_organization_name': 'Nature Portfolio', 'host_organization_lineage': ['https://openalex.org/P4310319908', 'https://openalex.org/P4310319965'], 'host_organization_lineage_names': ['Nature Portfolio', 'Springer Nature'], 'type': 'journal'}, 'license': 'cc-by', 'license_id': 'https://openalex.org/licenses/cc-by', 'version': 'publishedVersion', 'is_accepted': True, 'is_published': True, 'raw_

In [7]:
ast.literal_eval(final_df['primary_topic'][0])

{'id': 'https://openalex.org/T13650',
 'display_name': 'Computational Physics and Python Applications',
 'score': 0.9994999766349792,
 'subfield': {'id': 'https://openalex.org/subfields/1702',
  'display_name': 'Artificial Intelligence'},
 'field': {'id': 'https://openalex.org/fields/17',
  'display_name': 'Computer Science'},
 'domain': {'id': 'https://openalex.org/domains/3',
  'display_name': 'Physical Sciences'}}